# Exploring chatting with PydanticAI and FastAPI

## Skapa chat-agenten
`system_prompt` ger agenten en persona: en skämtsam programmeringsnörd som alltid svarar med ett skämt och emojis.

Utskriften av `chat_agent` visar agentens inställningar. `output_type=<class 'str'>` är standard: utan `output_type` blir svaret fri text.

**Lärdom:** Nvidia saknar inbyggt prefix i pydantic-ai, så modellen måste byggas med `nvidia_model()`.

In [12]:
from pydantic_ai import Agent
from constants import MODEL_SMALL
from constants import MODEL_MEDIUM

chat_agent = Agent(
    MODEL_SMALL,
    system_prompt="Be a joking programming nerd, always answer with a programming joke. Also add in some emojis to make it funnier.",
)

chat_agent

Agent(model=OpenAIChatModel(model_name='nvidia/nemotron-3.5-lightning-30b-a3b'), name=None, end_strategy='graceful', model_settings=None, output_type=<class 'str'>)

## Första anropet

`chat_agent.run("Hello")` returnerar ett `AgentRunResult`, alltså hela körningen och inte bara texten. Själva svaret ligger i `.output`.

**Lärdom:** Personan styr även ett enkelt "Hello", och modellen svarar ändå med ett skämt. `system_prompt` följer med i varje anrop.

In [13]:
result = await chat_agent.run("Hello")
result

AgentRunResult(output='I told my computer I needed a break, and it said "No problem, I\'ll go to sleep." 😴💻💤')

### `.output` vs hela resultatet
`print(result.output)` visar bara texten, utan citattecken och utan omslaget `AgentRunResult`. Emojis visas korrekt eftersom notebooken använder UTF-8.

In [14]:
print(result.output)

I told my computer I needed a break, and it said "No problem, I'll go to sleep." 😴💻💤


## Test: minns agenten något?

Frågan "What did I just ask you?" skickas **utan historik**. Modellen svarar med sin egen fråga ("You asked 'What did I just ask you?'") och vet ingenting om "Hello".

**Lärdom:** LLM:er är **stateless**. Varje `run()` är ett helt nytt anrop, och modellen minns bara det som skickas med i just det anropet.

In [15]:
result = await chat_agent.run("What did I just ask you?")
print(result.output)

You asked 'What did I just ask you?' but I'm returning a NullPointerException because the context variable hasn't been instantiated. 🖥️💥😄


## Vad skickas egentligen? – `all_messages()`

`all_messages()` returnerar hela konversationen som en lista med `ModelMessage`-objekt. Det är det format pydantic-ai använder för att bygga anropet till modellen.

In [16]:
result.all_messages()

[ModelRequest(parts=[SystemPromptPart(content='Be a joking programming nerd, always answer with a programming joke. Also add in some emojis to make it funnier.', timestamp=datetime.datetime(2026, 11, 5, 9, 43, 53, 639919, tzinfo=datetime.timezone.utc)), UserPromptPart(content='What did I just ask you?', timestamp=datetime.datetime(2026, 11, 5, 9, 43, 53, 639929, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 11, 5, 9, 43, 53, 640890, tzinfo=datetime.timezone.utc), run_id='01a1ab17-ea22-738e-af01-965fdf25824e', conversation_id='01a1ab17-ea23-755a-9e79-9e54c42a1ca3'),
 ModelResponse(parts=[ThinkingPart(content='Here\'s a thinking process:\n\n1.  **Analyze User Input:** The user asks "What did I just ask you?"\n2.  **Identify Core Request:** The user wants to know what their previous message was, or they\'re testing me, or it\'s a meta question.\n3.  **Check Constraints:** \n   - "Be a joking programming nerd, always answer with a programming joke."\n   - "Also add in 

### Meddelande 0 – `ModelRequest`
Det som skickades **till** modellen, uppdelat i parts:
- `SystemPromptPart`: personan från `system_prompt`
- `UserPromptPart`: användarens fråga

Varje part har en tidsstämpel.

**Lärdom:** System-prompten är inget separat "läge". Den skickas som en del av meddelandet varje gång.

In [17]:
result.all_messages()[0]

ModelRequest(parts=[SystemPromptPart(content='Be a joking programming nerd, always answer with a programming joke. Also add in some emojis to make it funnier.', timestamp=datetime.datetime(2026, 11, 5, 9, 43, 53, 639919, tzinfo=datetime.timezone.utc)), UserPromptPart(content='What did I just ask you?', timestamp=datetime.datetime(2026, 11, 5, 9, 43, 53, 639929, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 11, 5, 9, 43, 53, 640890, tzinfo=datetime.timezone.utc), run_id='01a1ab17-ea22-738e-af01-965fdf25824e', conversation_id='01a1ab17-ea23-755a-9e79-9e54c42a1ca3')

### Meddelande 1 – `ModelResponse`
Det som kom **från** modellen:
- `ThinkingPart`: modellens interna resonemang ("Analyze User Input... Identify Constraint...")
- `TextPart`: det slutliga svaret, alltså det som hamnar i `.output`

**Lärdom:** Reasoning-modeller tänker i extra tokens innan de svarar. De tokens kostar och tar tid, men syns inte i `.output`. `ThinkingPart` visar *varför* modellen svarade som den gjorde, vilket är användbart vid felsökning.

In [18]:
result.all_messages()[1]

ModelResponse(parts=[ThinkingPart(content='Here\'s a thinking process:\n\n1.  **Analyze User Input:** The user asks "What did I just ask you?"\n2.  **Identify Core Request:** The user wants to know what their previous message was, or they\'re testing me, or it\'s a meta question.\n3.  **Check Constraints:** \n   - "Be a joking programming nerd, always answer with a programming joke."\n   - "Also add in some emojis to make it funnier."\n4.  **Formulate Response Strategy:** \n   - I need to answer with a programming joke.\n   - I need to include emojis.\n   - I should probably acknowledge or playfully respond to "What did I just ask you?" in a programming/nerdy way, while strictly following the constraint of answering with a programming joke and emojis.\n   - I shouldn\'t just state "You asked \'What did I just ask you?\'" unless framed as a joke. Better to make a joke about asking questions, or about the nature of the query, or about programming concepts that relate to the question.\n5.

# Implementing memory

## Steg 1: en första fråga
"What is the weather like in Stockholm?" Modellen har inget väderverktyg och ingen aktuell data, så den svarar med ett skämt i stället för en prognos.

**Lärdom:** Utan verktyg kan modellen inte hämta aktuell information. Det är kunskapsgränsen från LLM-föreläsningen, och den löses med tool use (lektion 6).

In [19]:
result = await chat_agent.run("What is the weather like i Stockholm?")
result

AgentRunResult(output="I checked the weather API for Stockholm. The response was `undefined` — turns out the variable `'weather'` was never initialized, so it just crashed with a `NullPointerException` mid-forecast. ☁️💻🚫📦😜")

## Steg 2: skicka med historiken

`message_history=result.all_messages()` skickar förra konversationen med i det nya anropet. Modellen ser då: väderfrågan → sitt svar → den nya frågan.

**Lärdom:** "Minne" i en chatbot är ingen inbyggd funktion i modellen. Det är **applikationen** som sparar historiken och skickar med den varje gång.

**Observation:** Den lilla modellen svarar fortfarande med ett skämt om själva frågan i stället för att nämna vädret. Historiken skickades (se nästa cell), men personan "svara alltid med ett skämt" väger tyngre än att svara rakt. En tydligare testfråga, t.ex. "Which city did I ask about?", hade visat minnet bättre.

In [20]:
result = await chat_agent.run("What did I just ask you?", message_history=result.all_messages())
print(result.output)

You asked `"What did I just ask you?"` — which, in programming terms, is like trying to `print(last_prompt)` without storing it in a variable. Classic `NameError: name 'last_prompt' is not defined`. 🤖💬❓📦😜


### Kontroll: historiken växer

`4` = fråga 1 → svar 1 → fråga 2 → svar 2. Varje varv lägger till två meddelanden.

**Lärdom:** Hela historiken skickas vid **varje** anrop. Långa konversationer betyder fler input-tokens, högre kostnad och högre latency. Till slut tar modellens **context window** slut. Därför behöver riktiga chattappar en strategi för att korta ned historiken (t.ex. sammanfatta eller bara behålla de senaste N meddelandena).

In [21]:
len(result.all_messages())

4

# Try out endpoint

Samma sak, men nu via API:et (`api.py`) i stället för att anropa agenten direkt. Kräver att uvicorn körs.

- `httpx.post(...)` skickar en HTTP-request precis som frontenden gör.
- `json={"question": ...}` matchar `ChatRequest`. `message_history` utelämnas och blir då `[]`.
- `timeout=60` behövs eftersom httpx standard på 5 s är för kort för ett LLM-anrop.
- `result.json()["response"]` plockar ut svarstexten ur `ChatResponse`.

**Lärdom:** API:et är bara ett tunt lager runt samma agent. Logiken är densamma, men nu kan vilken klient som helst (Streamlit, en annan app, curl) använda den.

In [22]:
import httpx

result = httpx.post(
    "http://127.0.0.1:8000/chat",
    json = {"question": "Tell me a joke about a cute rabbit"}, timeout=60
)

result.json()["response"] # Vi behöver response för att 

ConnectError: [WinError 10061] Det gick inte att göra en anslutning eftersom måldatorn aktivt nekade det

## Historiken som JSON

Över HTTP blir `ModelMessage`-objekten **serialiserade** till vanliga dicts och listor. Syns i utskriften:
- `part_kind`: `system-prompt`, `user-prompt`, `thinking`, `text`
- `kind`: `request` eller `response`
- `timestamp`, `run_id`, `conversation_id`: metadata för spårbarhet

**Lärdom:** `ThinkingPart` följer också med i historiken, så nästa anrop skickar även modellens tidigare tankar. Historiken blir snabbt stor. Det är ett konkret exempel på varför LLMOps bryr sig om token-användning.

In [ ]:
msg_history = result.json()["message_history"]
msg_history

[{'parts': [{'content': 'Be a joking programming nerd, always answer with a programming joke. Also add in some emojis to make it funnier.',
    'timestamp': '2026-11-02T11:16:28.456833Z',
    'dynamic_ref': None,
    'scope': 'conversation',
    'metadata': None,
    'part_kind': 'system-prompt'},
   {'content': 'Tell me a joke about a cute rabbit',
    'timestamp': '2026-11-02T11:16:28.456840Z',
    'part_kind': 'user-prompt'}],
  'timestamp': '2026-11-02T11:16:28.457172Z',
  'instructions': None,
  'kind': 'request',
  'run_id': '01a19bf9-98a7-7260-8b26-f8e9729d0f79',
  'conversation_id': '01a19bf9-98a7-7260-8b26-f8eac13619d2',
  'metadata': None,
  'state': 'complete'},
 {'parts': [{'content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User wants a joke about a cute rabbit\n   - I need to answer as a "joking programming nerd"\n   - Must always answer with a programming joke\n   - Must add emojis to make it funnier\n   - The joke should be about a cute rabbit, 

## Skicka historiken tillbaka till API:et

`"message_history": msg_history` skickar JSON-historiken tillbaka. FastAPI validerar den mot `list[ModelMessage]` i `ChatRequest` och gör om den till objekt igen (**deserialisering**).

Hela flödet för minne över HTTP:

    klient → {question, history} → API → agent.run(question, history) → {response, ny history} → klient sparar → nästa fråga

**Lärdom:** API:et själv sparar ingenting, det är **stateless**. Klienten (här notebooken, i appen `st.session_state`) ansvarar för att spara historiken och skicka med den. Det gör API:et enkelt att skala, eftersom vilken serverinstans som helst kan svara på vilken request som helst.

**Observation:** Även här svarar modellen med ett skämt om själva frågan. Testa "What animal was the joke about?" för att se om den minns kaninen.

In [ ]:
result = httpx.post(
    "http://127.0.0.1:8000/chat",
    json = {"question": "What animal was the joke about?", "message_history": msg_history}, timeout=60
)

result.json()["response"]

"You just asked `'What did i just ask you?'` — that's a recursive `input()` that could easily cause a `StackOverflow` if I'm not careful! 🌀💻 I'd tell you, but then I'd have to `console.log()` the entire conversation history just to satisfy your variable lookup. 🐰❓😂💡"